# 02 — vmap and scan

Two JAX primitives that replace Python loops with compiled, hardware-efficient operations:

| Primitive | Replaces | Use case |
|-----------|----------|----------|
| `jax.vmap` | `for` loop over a **batch** dimension | same operation on many inputs simultaneously |
| `jax.lax.scan` | `for` loop with **sequential state** | RNNs, ODE solvers, cumulative operations |

Both are composable with `jit` and `grad`.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import time
import matplotlib.pyplot as plt

## 2.1 `vmap` — Vectorising over a Batch Dimension

Write a function for a **single example**, then `vmap` it over a batch — no manual broadcasting needed.

In [ ]:
# Single-example function
def linear_transform(W, x):
    """W: (m, n), x: (n,) → (m,)"""
    return W @ x

W = jnp.ones((4, 3))
x_single = jnp.array([1.0, 2.0, 3.0])
print('Single:   ', linear_transform(W, x_single))

# Batched: apply over 8 different x vectors
x_batch = jnp.ones((8, 3)) * jnp.arange(8)[:, None]

# Manual loop (slow in Python)
manual = jnp.stack([linear_transform(W, x_batch[i]) for i in range(8)])

# vmap (vectorised — no Python loop)
batch_transform = jax.vmap(linear_transform, in_axes=(None, 0))  # W fixed, batch over x
vmapped = batch_transform(W, x_batch)

print('Shapes — manual:', manual.shape, '  vmap:', vmapped.shape)
print('Results match:', jnp.allclose(manual, vmapped))

In [ ]:
# Speedup: vmap vs Python loop
key = jax.random.PRNGKey(0)
W_big = jax.random.normal(key, (64, 64))
X_big = jax.random.normal(key, (512, 64))   # 512 examples

batch_fn = jax.jit(jax.vmap(lambda x: W_big @ x))
_ = batch_fn(X_big).block_until_ready()  # warm up

n = 50
t0 = time.perf_counter()
for _ in range(n):
    r = jnp.stack([W_big @ X_big[i] for i in range(512)])
elapsed_loop = (time.perf_counter() - t0) / n * 1000

t0 = time.perf_counter()
for _ in range(n):
    batch_fn(X_big).block_until_ready()
elapsed_vmap = (time.perf_counter() - t0) / n * 1000

print(f'Python loop: {elapsed_loop:.2f} ms   |   vmap+jit: {elapsed_vmap:.2f} ms')
print(f'Speedup: {elapsed_loop/elapsed_vmap:.1f}x')

### Batch-wise Jacobian with double `vmap`

Computing per-example Jacobians is a common scientific computing task (e.g., neural tangent kernels, sensitivity analysis).

In [ ]:
def f(x):
    """x: (3,) → (3,)"""
    return jnp.array([jnp.sum(x**2), jnp.prod(x), jnp.sin(x[0])])

# Jacobian for a single input
J_single = jax.jacfwd(f)(jnp.array([1.0, 2.0, 3.0]))
print('Single Jacobian shape:', J_single.shape)   # (3, 3)

# Per-example Jacobian over a batch — using vmap over jacfwd
batch_jacobian = jax.jit(jax.vmap(jax.jacfwd(f)))
X = jax.random.normal(jax.random.PRNGKey(1), (16, 3))
Js = batch_jacobian(X)
print('Batch Jacobian shape:', Js.shape)  # (16, 3, 3)

## 2.2 `lax.scan` — Efficient Sequential Loops

```python
final_carry, stacked_outputs = jax.lax.scan(f, init_carry, xs)
```

- `f(carry, x) -> (new_carry, output)` — pure step function
- `init_carry` — initial state
- `xs` — sequence of inputs, shape `(T, ...)`; each step receives `xs[t]`

The loop is **unrolled once at trace time** into a single XLA `while` op — no Python overhead at runtime.

In [ ]:
# Example 1: cumulative sum
def cumsum_step(carry, x):
    new_carry = carry + x
    return new_carry, new_carry   # (carry, output_at_this_step)

xs = jnp.arange(1, 6, dtype=float)   # [1, 2, 3, 4, 5]
final, cumsum = jax.lax.scan(cumsum_step, 0.0, xs)
print('cumsum:', cumsum)         # [1, 3, 6, 10, 15]
print('final carry:', final)     # 15

In [ ]:
# Example 2: exponential moving average
def ema_step(carry, x, alpha=0.1):
    ema = (1 - alpha) * carry + alpha * x
    return ema, ema

signal = jnp.sin(jnp.linspace(0, 4*jnp.pi, 200)) + 0.3 * jax.random.normal(jax.random.PRNGKey(0), (200,))
_, smoothed = jax.lax.scan(ema_step, 0.0, signal)

plt.figure(figsize=(10, 3))
plt.plot(signal, alpha=0.4, label='noisy signal')
plt.plot(smoothed, lw=2, label='EMA (α=0.1)')
plt.legend()
plt.title('Exponential moving average via lax.scan')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Speedup: scan vs Python loop (the gap widens with sequence length)
T = 1000
xs_big = jax.random.normal(jax.random.PRNGKey(0), (T, 32))  # sequence of 32-dim vectors

@jax.jit
def scan_rnn(xs):
    W = jnp.eye(32) * 0.9   # simple recurrence
    def step(h, x):
        h = jnp.tanh(h @ W + x)
        return h, h
    _, hs = jax.lax.scan(step, jnp.zeros(32), xs)
    return hs

_ = scan_rnn(xs_big).block_until_ready()  # compile

n = 30
t0 = time.perf_counter()
for _ in range(n):
    W = jnp.eye(32) * 0.9
    h = jnp.zeros(32)
    for t in range(T): h = jnp.tanh(h @ W + xs_big[t])
elapsed_py = (time.perf_counter()-t0)/n*1000

t0 = time.perf_counter()
for _ in range(n): scan_rnn(xs_big).block_until_ready()
elapsed_scan = (time.perf_counter()-t0)/n*1000

print(f'Python loop: {elapsed_py:.1f} ms  |  scan+jit: {elapsed_scan:.2f} ms  |  speedup: {elapsed_py/elapsed_scan:.0f}x')

## 2.3 Carry Multiple Values and Emit Multiple Outputs

In [ ]:
# Carry can be any pytree (tuple, dict, nested)
def rnn_step_with_stats(carry, x):
    h, step_count = carry
    W = jnp.eye(h.shape[0]) * 0.9
    h_new = jnp.tanh(W @ h + x)
    new_carry = (h_new, step_count + 1)
    output = {'hidden': h_new, 'norm': jnp.linalg.norm(h_new)}
    return new_carry, output

T, D = 50, 8
xs = jax.random.normal(jax.random.PRNGKey(0), (T, D))
init = (jnp.zeros(D), jnp.array(0))
(h_final, n_steps), outputs = jax.lax.scan(rnn_step_with_stats, init, xs)

print('h_final shape:', h_final.shape)
print('n_steps:', n_steps)
print('all hidden states shape:', outputs['hidden'].shape)
print('norm over time shape:', outputs['norm'].shape)

## 2.4 Memory-Efficient Backprop: `jax.checkpoint`

By default, JAX stores all intermediate activations for backprop (memory ∝ T). `jax.checkpoint` (aka `remat`) recomputes them on the backward pass instead, trading compute for memory — essential for long sequences.

In [ ]:
def rnn_loss(params, xs):
    W, b = params

    @jax.checkpoint  # recompute activations during backward pass
    def step(h, x):
        h_new = jnp.tanh(W @ h + b + x)
        return h_new, h_new

    _, hs = jax.lax.scan(step, jnp.zeros(W.shape[0]), xs)
    return jnp.mean(hs ** 2)   # dummy loss

D = 16
params = (jnp.eye(D) * 0.9, jnp.zeros(D))
xs = jax.random.normal(jax.random.PRNGKey(0), (500, D))

loss, grads = jax.value_and_grad(rnn_loss)(params, xs)
print('Loss:', float(loss))
print('∂L/∂W shape:', grads[0].shape)
print('∂L/∂b shape:', grads[1].shape)

## 2.5 Composing vmap + scan

Run an RNN scan over a **batch** of sequences simultaneously — the JAX way to handle batched sequential data.

In [ ]:
def run_rnn_single(W, xs):
    """Run RNN on one sequence (T, D)."""
    def step(h, x):
        return jnp.tanh(W @ h + x), jnp.tanh(W @ h + x)
    _, hs = jax.lax.scan(step, jnp.zeros(W.shape[0]), xs)
    return hs

# Batch over sequences (B, T, D)
run_rnn_batch = jax.jit(jax.vmap(run_rnn_single, in_axes=(None, 0)))

D, T, B = 16, 100, 32
W = jnp.eye(D) * 0.9
xs_batch = jax.random.normal(jax.random.PRNGKey(0), (B, T, D))

hs_batch = run_rnn_batch(W, xs_batch)
print('Output shape (B, T, D):', hs_batch.shape)  # (32, 100, 16)